<a href="https://colab.research.google.com/github/aritrasinha531-ai/silver-invention/blob/main/code_on_ANN_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
uploaded

In [ ]:
df=pd.read_csv("powerplant_data.csv")
df

In [ ]:
x=df.drop("PE",axis=1)
y=df["PE"]

In [ ]:
# split our data
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
import torch
import torch.nn as nn
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [ ]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [ ]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self).__init__()
    self.model=nn.Sequential(
    # 1st hidden layer
    nn.Linear(x_train.shape[1],6),
    nn.ReLU(),
    # 2nd hidden layer
    nn.Linear(6,6),
    nn.ReLU(),
    # output
    nn.Linear(6,1)
    )

  def forward(self,x):
    return self.model(x)

In [ ]:
def forward(self,x):
  return self.model(x)

In [ ]:
import torch.optim as optim
model=ANN()
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [ ]:
# train model
train_losses=[]
best_val_loss=float("inf")
valid_losses=[]
epochs=100
for epoch in range(epochs):
  model.train()
  running_loss=0.0
  for xb,yb in train_loader:
    optimizer.zero_grad()

    outputs=model(xb)
    loss=criterion(outputs,yb)
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()

In [ ]:
epoch_train_loss=running_loss/len(train_loader)
train_losses.append(epoch_train_loss)

In [ ]:
# validation
model.eval()
running_val_loss=0.0
with torch.no_grad():
  for xb,yb in test_loader:
    outputs=model(xb)
    loss=criterion(outputs,yb)
    running_val_loss+=loss


In [ ]:
epoch_val_loss=running_val_loss/len(test_loader)
valid_losses.append(epoch_val_loss)
print(f"epoch{epoch+1}/{epochs}=>train loss={epoch_train_loss} & val loss={epoch_val_loss}")

In [ ]:
# evaluation
model.eval()
with torch.no_grad():
  train_preds=model(x_train_tensor)
  test_preds=model(x_test_tensor)
  train_mse_loss=criterion(train_preds,y_train_tensor)
  test_mse_loss=criterion(test_preds,y_test_tensor)
  print("training mse=",train_mse_loss.item())
  print("testing mse=",test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score
print("r^2 score=",r2_score(y_test,test_preds))

In [ ]:
predicted_df=pd.DataFrame(test_preds.numpy(),columns=["predicted values"])
actual_df=pd.DataFrame(y_test.values,columns=["actual values"])
result_df=pd.concat([actual_df,predicted_df],axis=1)

In [ ]:
result_df

In [ ]:
print("end of project")